# 20. 일반화 개선전략 비교

baseline, photometric augmentation, group-balanced sampling을 같은 평가셋에서 비교합니다.
결론은 평균 Dice와 worst-group Dice를 함께 보고 냅니다.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "ch2_utils.py").exists():
    matches = (
        list(Path.cwd().glob("Deeplearning/*/2-1장/ch2_utils.py"))
        + list(Path.cwd().glob("Deeplearning/*/2장/ch2_utils.py"))
        + list(Path.cwd().glob("**/ch2_utils.py"))
    )
    NOTEBOOK_DIR = matches[0].parent if matches else Path("Deeplearning") / "Vision 응용" / "2-1장"
sys.path.append(str(NOTEBOOK_DIR))

from ch2_utils import *

paths = find_paths()
set_korean_font()
set_seed(7)
samples = load_samples(paths.data_root)
paths

## 20-1. 전략별 manifest와 설정

In [ ]:
standard_train, standard_eval = create_standard_manifests(samples, paths.runs_root)
balanced_train = create_group_balanced_train_manifest(
    samples,
    out_path=paths.runs_root / "manifests" / "balanced" / "train_manifest.csv",
    group_cols=("color_group", "defect_type"),
    train_size=360,
    seed=7,
)

strategies = [
    {"name": "baseline_no_aug", "train_manifest": standard_train, "augment": False},
    {"name": "photometric_aug", "train_manifest": standard_train, "augment": "photometric"},
    {"name": "group_balanced", "train_manifest": balanced_train, "augment": False},
]
display(pd.DataFrame(strategies))

## 20-2. 전략별 학습과 추론

In [ ]:
strategy_rows = []
for strategy in strategies:
    run_dir = paths.runs_root / "improvement_b0" / strategy["name"]
    if not (run_dir / "sample_metrics.csv").exists():
        print("training:", strategy["name"])
        train_segformer_experiment(
            train_manifest=strategy["train_manifest"],
            eval_manifest=standard_eval,
            run_dir=run_dir,
            epochs=2,
            batch_size=8,
            lr=1e-3,
            seed=7,
            augment=strategy["augment"],
            model_name=SEGFORMER_B0_MODEL_NAME,
            use_pretrained=True,
        )
    sample_m, group_m, class_m = load_run_metrics(run_dir)
    color_groups = group_m[group_m["grouping"] == "color_group"]
    combo_groups = group_m[group_m["grouping"] == "color_group x shape_group x defect_type"]
    strategy_rows.append(
        {
            "strategy": strategy["name"],
            "mean_target_dice": sample_m["target_dice"].mean(),
            "mean_target_fnr": sample_m["target_fnr"].mean(),
            "worst_color_dice": color_groups["target_dice_mean"].min(),
            "worst_combo_dice": combo_groups["target_dice_mean"].min(),
        }
    )
strategy_df = pd.DataFrame(strategy_rows).sort_values("worst_combo_dice", ascending=False)
strategy_df.to_csv(paths.runs_root / "improvement_strategy_summary_b0.csv", index=False, encoding="utf-8-sig")
display(strategy_df)

## 20-3. 개선전략 비교 그래프

In [ ]:
ax = strategy_df.set_index("strategy")[["mean_target_dice", "worst_combo_dice"]].plot(kind="bar", figsize=(8, 4))
ax.set_ylim(0, 1)
ax.set_ylabel("Dice")
ax.set_title("개선전략별 평균 성능과 worst-combo 성능")
ax.grid(axis="y", alpha=0.25)
fig = ax.get_figure()
fig.tight_layout()
fig.savefig(paths.runs_root / "20_improvement_strategy_comparison_b0.png", dpi=150)
plt.show()

## 20-4. 개선전략 결론

In [ ]:
best = strategy_df.iloc[0]
print(
    f"결론: worst color-shape-defect 조합 기준으로 가장 좋은 전략은 {best['strategy']}입니다. "
    f"평균 Dice={best['mean_target_dice']:.3f}, worst-combo Dice={best['worst_combo_dice']:.3f}입니다."
)